In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

from pathlib import Path
from firthlogist import FirthLogisticRegression
from sklearn.utils.validation import check_X_y

# Opening cleaned PAROS dataset

In [2]:
CURRENT_DIRECTORY = Path.cwd().resolve()

# Find project root that contains datasets
PROJECT_ROOT = next(
    p for p in [CURRENT_DIRECTORY, *CURRENT_DIRECTORY.parents]
    if (p / "datasets").exists()
)

CLEANED_DATASET_PATH = PROJECT_ROOT / "datasets" / "PAROS_Dataset_Cleaned.csv"

if not CLEANED_DATASET_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEANED_DATASET_PATH}")

df = pd.read_csv(CLEANED_DATASET_PATH)
print(f"Loaded cleaned PAROS dataset: {df.shape}")
display(df.head(3))


Loaded cleaned PAROS dataset: (2076, 71)


,Patient brought in by,Date of Incident,Location of incident,Location Unknown,Location Type,Location Type Other,Age,Age Modifier,Gender,Race,...,Outcome of patient,Patient status,Date of Discharge or Death,Patient neurological status - Cerebral,Patient neurological status - Overall,Patient neurological status - Unknown,Year,Call_Time,Shock_Time,Time_to_Defib
0,Ems,2014-01-01,238889.0,NaN,Transport Center,Dhoby Ghaut Mrt Level B1,59,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-05 22:28:12,2026-04-05 22:39:17,11.083333
1,Ems,2014-01-05,272018.0,NaN,Public/Commercial Building,Level 2,66,Years,Male,Chinese,...,Died In Ed,NaN,NaN,5.0,NaN,NaN,2014,2026-04-05 15:00:42,2026-04-05 15:16:49,16.116667
2,Ems,2014-01-07,760105.0,NaN,Street/Highway,Level 1,80,Years,Male,Indian,...,Admitted,Remains In Hospital At 30Th Day Post Arrest,NaN,4.0,4.0,NaN,2014,2026-04-05 12:05:46,2026-04-05 12:14:08,8.366667


# Standardize Reporting

In [3]:
import pandas as pd
import numpy as np

def report_results(model_output, model_name="Model"):
    """
    Standardizes the reporting style for all versions.
    Prevents length-mismatch errors by building segments before joining.
    """
    print(f"\n{'='*20} {model_name} Results {'='*20}")
    
    # ---------------------------------------------------------
    # PATH A: Firthlogist Model (V3 & V4)
    # ---------------------------------------------------------
    if "firthlogist" in str(type(model_output)):
        # 1. Extract feature names and data
        features = list(getattr(model_output, 'feature_names_in_', []))
        if not features:
            features = [f'x{i+1}' for i in range(len(model_output.coef_))]
            
        # 2. Build feature table
        df_feats = pd.DataFrame(index=features)
        df_feats['Odds Ratio'] = np.exp(model_output.coef_)
        df_feats['Lower 95% CI'] = np.exp(model_output.ci_[:len(features), 0])
        df_feats['Upper 95% CI'] = np.exp(model_output.ci_[:len(features), 1])
        df_feats['p-value'] = model_output.pvals_[:len(features)]
        
        # 3. Build Intercept row separately
        # We use getattr with defaults to handle inconsistent naming in the library
        int_pval = getattr(model_output, 'intercept_pval_', getattr(model_output, 'intercept_pvals_', np.nan))
        # Handle cases where intercept_pval_ might be an array
        if isinstance(int_pval, (np.ndarray, list)): int_pval = int_pval[0]
            
        # Get Intercept CIs
        ci_int = getattr(model_output, 'ci_intercept_', [np.nan, np.nan])
        if hasattr(ci_int, 'ndim') and ci_int.ndim > 1: ci_int = ci_int.flatten()
        
        df_int = pd.DataFrame({
            'Odds Ratio': [np.exp(model_output.intercept_)],
            'Lower 95% CI': [np.exp(ci_int[0])],
            'Upper 95% CI': [np.exp(ci_int[1])],
            'p-value': [int_pval]
        }, index=['Intercept'])
        
        return pd.concat([df_feats, df_int])

    # ---------------------------------------------------------
    # PATH B: Statsmodels Model (V1 & V2)
    # ---------------------------------------------------------
    elif hasattr(model_output, 'summary2'):
        summary = model_output.summary2().tables[1].copy()
        summary.loc[:, 'OR'] = np.exp(summary['Coef.'])
        summary.loc[:, 'Lower_95_CI'] = np.exp(summary['Coef.'] - 1.96 * summary['Std.Err.'])
        summary.loc[:, 'Upper_95_CI'] = np.exp(summary['Coef.'] + 1.96 * summary['Std.Err.'])
        
        final_table = summary[['OR', 'Lower_95_CI', 'Upper_95_CI', 'P>|z|']]
        final_table.columns = ['Odds Ratio', 'Lower 95% CI', 'Upper 95% CI', 'p-value']
        return final_table
        
    else:
        raise ValueError("Unrecognized model object.")

# Define Binary Outcome (Survival to 30 Days)


In [4]:
outcome_cols = ['Outcome of patient', 'Patient status', 'Final status at scene']
df.loc[:, 'Outcome_String'] = df[outcome_cols].astype(str).agg(' '.join, axis=1)
survival_regex = r'Discharged Alive|Remains in hospital at 30th day'
df.loc[:, 'Survival_Binary'] = df['Outcome_String'].str.contains(survival_regex, case=False, regex=True).astype(int)

# Define Primary Predictor (Bystander AED applied)
- We map any variation of 'Yes' or '1' to 1, and everything else to 0

In [5]:
aed_col = 'Bystander AED applied'
df.loc[:, 'AED_Applied_Binary'] = df[aed_col].astype(str).str.contains('Yes|Applied|1', case=False, na=False).astype(int)

# Define Adjustment Variables(Multivariable Context)

In [6]:
df.loc[:, 'Age'] = pd.to_numeric(df['Age'], errors='coerce')

witness_pattern = 'Bystander|Ems'
df.loc[:, 'Witnessed_Binary'] = df['Arrest witnessed by'].astype(str).str.contains(witness_pattern, case=False, na=False).astype(int)
df.loc[:, 'Bystander_CPR_Binary'] = df['Bystander CPR'].astype(str).str.contains('Yes|1', case=False, na=False).astype(int)
df.loc[:, 'Gender_Male'] = (df['Gender'].astype(str).str.strip().str.title() == 'Male').astype(int)

In [7]:
print("Witnessed Breakdown:")
print(df['Witnessed_Binary'].value_counts())

Witnessed Breakdown:
Witnessed_Binary
1    1529
0     547
Name: count, dtype: int64


# Cleaned DataFrame

In [8]:
model_vars = ['Survival_Binary', 'AED_Applied_Binary', 'Time_to_Defib', 'Age', 'Witnessed_Binary', 'Bystander_CPR_Binary', 'Gender_Male']
df_clean = df.dropna(subset=model_vars).copy()

# Grid Search

In [9]:
def run_grid_iteration(data, tau):
    t_before = data['Time_to_Defib'].clip(upper=tau)
    t_after = (data['Time_to_Defib'] - tau).clip(lower=0)
    
    X = pd.DataFrame({
        'const': 1.0,
        'AED': data['AED_Applied_Binary'],
        'Time_Before': t_before,
        'Time_After': t_after,
        'AED_x_Before': data['AED_Applied_Binary'] * t_before,
        'AED_x_After': data['AED_Applied_Binary'] * t_after,
        'Age': data['Age'],
        'Witnessed': data['Witnessed_Binary'],
        'Bystander_CPR': data['Bystander_CPR_Binary'],
        'Gender_Male': data['Gender_Male']
    })
    
    try:
        # Use BFGS for the search because it easily calculates AIC
        model = sm.Logit(data['Survival_Binary'], X).fit(method='bfgs', maxiter=500, disp=0)
        
        # Ensure model converged properly before trusting the AIC
        if not model.mle_retvals['warnflag'] == 0:
            return np.inf
        return model.aic
    except:
        return np.inf

# Check if we have survivors at late time points


In [10]:
late_survivors = df[(df['Time_to_Defib'] > 10) & (df['Survival_Binary'] == 1)]
print(f"Survivors after 10 mins: {len(late_survivors)}")
print(late_survivors['Bystander AED applied'].value_counts())

Survivors after 10 mins: 262
Bystander AED applied
No     243
Yes     19
Name: count, dtype: int64


# Calculate Empirical P-Values and CIs

In [11]:
search_results = []

print("\n--- Phase 1: AIC Grid Search for Optimal Threshold ---")
for t in range(3, 13):
    aic = run_grid_iteration(df_clean, t)
    if aic != np.inf:
        search_results.append({'tau': t, 'AIC': aic})
        print(f"Minute {t}: AIC = {aic:.2f}")

# Dynamically extract the winning threshold
discovery_df = pd.DataFrame(search_results)
best_tau = discovery_df.loc[discovery_df['AIC'].idxmin(), 'tau']
print(f"\n=> Validated Optimal Breakpoint (tau): {best_tau} minutes")


--- Phase 1: AIC Grid Search for Optimal Threshold ---
Minute 3: AIC = 1775.35
Minute 4: AIC = 1775.20
Minute 5: AIC = 1775.66
Minute 6: AIC = 1775.49
Minute 7: AIC = 1775.94
Minute 8: AIC = 1776.69
Minute 9: AIC = 1777.92
Minute 10: AIC = 1778.52
Minute 11: AIC = 1778.95
Minute 12: AIC = 1779.20

=> Validated Optimal Breakpoint (tau): 4 minutes


# Adjusted Piecewise Model (Firth)

In [12]:
t_before = df_clean['Time_to_Defib'].clip(upper=best_tau)
t_after = (df_clean['Time_to_Defib'] - best_tau).clip(lower=0)

X_final = pd.DataFrame({
    'AED': df_clean['AED_Applied_Binary'],
    'Time_Before': t_before,
    'Time_After': t_after,
    'AED_x_Before': df_clean['AED_Applied_Binary'] * t_before,
    'AED_x_After': df_clean['AED_Applied_Binary'] * t_after,
    'Age': df_clean['Age'],
    'Witnessed': df_clean['Witnessed_Binary'],
    'Bystander_CPR': df_clean['Bystander_CPR_Binary'],
    'Gender_Male': df_clean['Gender_Male']
})
y_final = df_clean['Survival_Binary']

def patched_validate(self, X, y, **kwargs):
    return check_X_y(X, y, **kwargs)

FirthLogisticRegression._validate_data = patched_validate

# Fit the penalized Firth model
firth_model = FirthLogisticRegression(max_iter=1000)
firth_model.feature_names_in_ = X_final.columns 
firth_model.fit(X_final, y_final)

# 2. Extract coefficients
coefs = np.array(firth_model.coef_).flatten()
intercept = float(firth_model.intercept_[0]) if isinstance(firth_model.intercept_, (list, np.ndarray)) else float(firth_model.intercept_)
linear_pred = np.dot(X_final, coefs) + intercept

# 3. Calculate pure probabilities
p = 1 / (1 + np.exp(-linear_pred))
p = np.clip(p, 1e-15, 1 - 1e-15)

# 4. Calculate Unpenalized AIC
unpenalized_ll = np.sum(y_final * np.log(p) + (1 - y_final) * np.log(1 - p))
k = X_final.shape[1] + 1 
aic = 2 * k - 2 * unpenalized_ll

# Report

In [14]:
v4_report = report_results(firth_model, f"Version 4: Adjusted Piecewise (tau={best_tau})")
display(v4_report)


==================== Version 4: Adjusted Piecewise (tau=4) Results ====================


,Odds Ratio,Lower 95% CI,Upper 95% CI,p-value
AED,0.007377,2.616833e-09,5.174327,1.561811e-01
Time_Before,0.598846,2.016612e-02,2.221324,4.781612e-01
Time_After,0.871048,8.341232e-01,0.908850,9.404300e-11
AED_x_Before,3.193035,5.876160e-01,136.193216,1.997896e-01
AED_x_After,1.058440,9.379246e-01,1.190158,3.500651e-01
Age,0.967651,9.593027e-01,0.975982,3.830867e-14
Witnessed,1.430180,1.074185e+00,1.925097,1.383667e-02
Bystander_CPR,1.542800,1.165611e+00,2.061575,2.244524e-03
Gender_Male,1.158239,8.224302e-01,1.661834,4.052896e-01
Intercept,17.592409,NaN,NaN,NaN
